# GraphRAG Pipeline

Modular knowledge graph construction and retrieval for engineering documents.

This notebook is a thin driver that imports from the `graphrag/` package.
Each pipeline step is one cell — customize by editing `graphrag/config.py`.

## 0. Setup & Configuration

In [1]:
from graphrag.connections import init_connections
import graphrag.config as config

driver, emb_model, llm = init_connections(config)

c:\Users\hler\AppData\Local\anaconda3\envs\dat300\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Neo4j driver initialized (bolt://localhost:7687)
Embedding model: qwen3-embedding:8b
LLM: gpt-5.4-mini (Azure)


## 1. Document Ingestion

Load documents from all enabled formats, extract metadata, segment into sections,
group lists, and merge text.

In [ ]:
documents = []

if "docx" in config.DOCUMENT_FORMATS:
    from graphrag.ingestion.docx_loader import load_and_segment
    documents.extend(load_and_segment(config, llm))

if "pdf" in config.DOCUMENT_FORMATS:
    from graphrag.pdf_ingestion.pdf_loader import load_and_segment as load_pdf
    documents.extend(load_pdf(config, llm))

print(f"\nTotal: {len(documents)} documents loaded")
for doc in documents:
    n_sec = len(doc.get('sections', []))
    print(f"  {doc['source_file']}  ({n_sec} sections)")

Loading 32001-J-RA-0001-01-1_.DOCX
   163 blocks | filename: 6 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 10 | 1.4s
Loading 32002-J-RA-0001-01-1_INCLUDING COMMENTS CAST.DOCX
   402 blocks | filename: 7 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 26 | 1.4s
Loading 32003-J-RA-0001-02-1_.DOCX
   119 blocks | filename: 6 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 9 | 1.3s
Loading 32003-J-RA-0002-01-1_.DOCX
   96 blocks | filename: 6 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 9 | 1.0s
Loading 32003-J-RA-0003-01-1_.DOCX
   32 blocks | filename: 6 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 9 | 1.0s
Loading 32003-J-RA-0004-01-3_report.DOCX
   95 blocks | filename: 7 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 7 | 1.0s
Loading 32003-J-RA-0005-01A-1_.DOCX
   34 blocks | filename: 6 fields | LLM: ['date', 'rig', 'odt_reference_persons'] | abbrevs: 5 | 0.9s
Loadin

## 1b. PDF Processing Inspection

These cells inspect the PDF extraction quality. Only relevant when PDFs are loaded.
Run after Document Ingestion to verify header/footer filtering, text repair, and table extraction.

In [3]:
# ── Processing Stats Summary ─────────────────────────────────────────────
pdf_docs = [d for d in documents if d["source_file"].lower().endswith(".pdf")]

if pdf_docs:
    print(f"{'Document':<45} {'Pages':>5} {'Text':>5} {'List':>5} {'Table':>5} {'Repaired':>8} {'Reverted':>8} {'Sections':>8} {'HF Filt':>7}")
    print("-" * 120)
    for d in pdf_docs:
        s = d.get("_pdf_processing_stats", {})
        print(
            f"{d['source_file'][:44]:<45} "
            f"{s.get('pages', '?'):>5} "
            f"{s.get('blocks_text', '?'):>5} "
            f"{s.get('blocks_list', '?'):>5} "
            f"{s.get('blocks_table', '?'):>5} "
            f"{s.get('repairs_applied', '?'):>8} "
            f"{s.get('repairs_reverted', '?'):>8} "
            f"{s.get('sections', '?'):>8} "
            f"{s.get('headers_filtered', '?'):>7}"
        )
        if s.get("tables_failed", 0) > 0:
            print(f"  ^ WARNING: {s['tables_failed']}/{s['tables_detected']} tables failed extraction")
else:
    print("No PDF documents loaded — skipping PDF inspection.")

No PDF documents loaded — skipping PDF inspection.


In [4]:
# ── Header/Footer Inspection ─────────────────────────────────────────────
# Shows what text was filtered as headers/footers and how often it appeared.
# Review this to check for false positives (real content incorrectly removed).

for d in pdf_docs:
    details = d.get("_pdf_header_footer_details", [])
    if not details:
        print(f"\n{d['source_file']}: No headers/footers filtered")
        continue
    print(f"\n{d['source_file']} — {len(details)} patterns filtered:")
    print(f"  {'Text':<70} {'Pages':>5} {'Fraction':>8}")
    print(f"  {'-'*85}")
    for item in details[:25]:  # show top 25
        text_preview = item['text'][:69]
        print(f"  {text_preview:<70} {item['count']:>5} {item['fraction']:>8.0%}")
    if len(details) > 25:
        print(f"  ... and {len(details) - 25} more")

In [5]:
# ── Text Repair Samples ──────────────────────────────────────────────────
# Shows before/after for repaired text blocks so you can verify quality.
# Also flags any repairs that were reverted (LLM lost >20% of words).

MAX_SAMPLES = 5  # Number of repair samples to show per document

for d in pdf_docs:
    log = d.get("_pdf_repair_log", [])
    if not log:
        print(f"\n{d['source_file']}: No text repairs logged")
        continue

    applied = [r for r in log if r["status"] == "applied"]
    reverted = [r for r in log if r["status"] == "reverted"]
    print(f"\n{d['source_file']} — {len(applied)} applied, {len(reverted)} reverted")

    if reverted:
        print(f"\n  REVERTED REPAIRS (LLM dropped too many words):")
        for r in reverted[:3]:
            print(f"    Reason: {r['reason']}")
            print(f"    Original:  {r['original'][:120]}...")
            print(f"    Repaired:  {r['repaired'][:120]}...")
            print()

    if applied:
        print(f"  APPLIED REPAIRS (sample of {min(MAX_SAMPLES, len(applied))}):")
        for r in applied[:MAX_SAMPLES]:
            print(f"    BEFORE: {r['original'][:150]}")
            print(f"    AFTER:  {r['repaired'][:150]}")
            print()

In [6]:
# ── Table Extraction Check ───────────────────────────────────────────────
# Shows a sample of successfully extracted tables per document.
# Check that tables have sensible headers and data.

MAX_TABLE_SAMPLES = 3  # Tables to preview per document

for d in pdf_docs:
    s = d.get("_pdf_processing_stats", {})
    tables = [b for b in d.get("blocks", []) if b["type"] == "table"]
    print(f"\n{d['source_file']} — {len(tables)} tables extracted"
          + (f" ({s.get('tables_failed', 0)} failed)" if s.get("tables_failed") else ""))

    for i, t in enumerate(tables[:MAX_TABLE_SAMPLES]):
        rows = t["rows"]
        print(f"\n  Table {i+1} ({len(rows)} rows, {len(rows[0]) if rows else 0} cols):")
        for row in rows[:4]:  # show header + first 3 data rows
            cells = [c[:25] for c in row]  # truncate wide cells
            print(f"    | {' | '.join(cells)} |")
        if len(rows) > 4:
            print(f"    ... ({len(rows) - 4} more rows)")
    if len(tables) > MAX_TABLE_SAMPLES:
        print(f"\n  ... and {len(tables) - MAX_TABLE_SAMPLES} more tables")

## 2. Chunking

Split documents into retrieval-ready chunks. Chunks are not embedded —
retrieval reaches them through the entity graph.

In [7]:
from graphrag.chunking import chunk_all

chunk_all(documents, cfg=config)

# Summary
total = sum(len(d.get('chunks', [])) for d in documents)
by_type = {}
for d in documents:
    for c in d.get('chunks', []):
        by_type[c['chunk_type']] = by_type.get(c['chunk_type'], 0) + 1
print(f"\n{total} chunks total: {dict(by_type)}")

Chunked 478 documents (15526 total chunks)

15526 chunks total: {'table': 4365, 'text': 9366, 'list': 1795}


## 3. Neo4j Graph Construction

Write Document, Section, and Chunk nodes with relationships.
Handles revision-aware supersession automatically.

In [9]:
from graphrag.graph.ingestion import ingest_all_documents

results = ingest_all_documents(documents, driver)

print(f"\nIngested: {len(results['ingested'])}")
print(f"Superseded: {len(results['superseded'])}")
print(f"Skipped: {len(results['skipped'])}")

Constraints and indexes ready
  INGEST    32001-J-RA-0001-01-1_.DOCX
  INGEST    32002-J-RA-0001-01-1_INCLUDING COMMENTS CAST.DOCX
  INGEST    32003-J-RA-0001-02-1_.DOCX
  INGEST    32003-J-RA-0002-01-1_.DOCX
  INGEST    32003-J-RA-0003-01-1_.DOCX
  INGEST    32003-J-RA-0004-01-3_report.DOCX
  INGEST    32003-J-RA-0005-01A-1_.DOCX
  INGEST    32003-J-RA-0006-01A-1_.DOCX
  INGEST    32003-J-RA-0007-02-1_.DOCX
  INGEST    32003-J-RA-0008-01-1_Report.DOCX
  INGEST    32003-J-RA-0009-DIC-1_DRIVE-OFF ANALYSIS - DSY.DOCX
  INGEST    32003-J-RA-0010-01-1_Rapport.DOCX
  INGEST    32003-J-RA-0011--1_DSY Drift-off & Drive-off Calculator Report.DOCX
  INGEST    32003-J-RA-0012-02A-1_Report.DOCX
  INGEST    32003-J-RA-0013-02A-1_SEE PROJECT NO 34900.DOCX
  INGEST    32003-J-RA-0014-01A-1_.DOCX
  INGEST    32003-J-RA-0015-01-1_.DOCX
  INGEST    32003-J-RA-0016-01-2_.DOCX
  INGEST    32003-J-RA-0017-01-1_.DOCX
  INGEST    32007-N-RA-0002-01-1_.DOCX
  INGEST    32011-J-RA-0001-01-1_.DOCX
  INGEST    

In [10]:
# Create the community embedding vector index (run once)
# Only Community nodes carry embeddings in this pipeline — chunks are reached
# through the entity graph.
from graphrag.graph.schema import create_vector_indexes

embedding_dim = len(emb_model.embed_query("dimension probe"))
create_vector_indexes(driver, embedding_dim)

  Vector index 'community_embedding_index' already exists


## 4. Entity Extraction

Extract typed entities from all chunks using LLM + schema guidance.
Applies canonicalization and filtering per chunk type.

In [ ]:
from graphrag.entities.orchestrator import extract_and_write_all_entities

entity_stats = extract_and_write_all_entities(
    documents, driver, llm, cfg=config,
)

print(f"\nChunks processed: {entity_stats['chunks_processed']}")
print(f"Chunks skipped:  {entity_stats['chunks_skipped']}")
print(f"Entities extracted: {entity_stats['entities_extracted']}")
print(f"Entities written:  {entity_stats['entities_written']}")
print(f"Relationships extracted: {entity_stats['relationships_extracted']}")
print(f"Relationships written:  {entity_stats['relationships_written']}")

## 5. Community Detection & Summarization

Detect entity communities using the Leiden algorithm (via Neo4j GDS).
Each community gets an LLM-generated summary and embedding for retrieval.

In [12]:
import importlib
import graphrag.communities.detection
importlib.reload(graphrag.communities.detection)
from graphrag.communities.detection import detect_communities

from graphrag.graph.schema import create_community_constraints
from graphrag.communities.detection import detect_communities
from graphrag.communities.summarization import summarize_communities

# Create community constraints
with driver.session() as session:
    session.execute_write(create_community_constraints)

# Detect communities via Leiden algorithm (requires Neo4j GDS plugin)
community_stats = detect_communities(driver)
print(f"Communities found: {community_stats['community_count']}")
print(f"Modularity: {community_stats['modularity']:.4f}")
print(f"Top 10 community sizes: {dict(list(community_stats['sizes'].items())[:10])}")

# Generate LLM summaries + embeddings for each community
summary_stats = summarize_communities(driver, llm, emb_model)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('entity-community-graph', false)"


Community constraints ready
  Projecting 7 relationship types: ASSOCIATED_WITH, CLIENT_OF, COMPLIES_WITH, LOCATED_AT, PROVIDES, SUPPLIER_FOR, WORKS_FOR
  GDS graph projected
  Leiden: 9905 communities, modularity=0.8384
  Cleared community_id from 9740 entities in communities below size 2


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('entity-community-graph', false)"


Communities found: 165
Modularity: 0.8384
Top 10 community sizes: {351: 200, 3923: 81, 2884: 65, 2654: 61, 7729: 49, 3779: 45, 6671: 40, 5741: 32, 7659: 24, 2524: 24}
Summarizing 165 communities (>= 2 members)...
  Summarized 165 communities, skipped 0 (388.8s)


## 6. Query & Retrieval

Ask questions against the knowledge graph. Pure GraphRAG retrieval: query
terms match Entity nodes, typed-relationship and SIMILAR_TO traversal yield
scored chunks, and community summaries (local + semantic) provide cross-cluster
context.

In [13]:
from graphrag.retrieval.ask import ask

question = "What is the document number for the drift-off analysis performed on Deepsea Nordkapp?"

answer, metadata = ask(
    question,
    driver=driver,
    embed_fn=emb_model.embed_query,
    llm=llm,
    cfg=config,
)

print(f"\nAnswer:\n{answer}")
print(f"\nDocuments used: {metadata['identified_documents']}")
print(f"Chunks retrieved: {metadata['n_chunks']}")

[Stage 1] Document identification (medium confidence):
  -> 32003-J-RA-0007-02-1_.DOCX
  -> 32003-J-RA-0009-DIC-1_DRIVE-OFF ANALYSIS - DSY.DOCX
  -> 32003-J-RA-0006-01A-1_.DOCX
  -> 32003-J-RA-0002-01-1_.DOCX
  -> 32003-J-RA-0013-02A-1_SEE PROJECT NO 34900.DOCX
  -> 32003-J-RA-0015-01-1_.DOCX
  -> 32003-J-RA-0010-01-1_Rapport.DOCX
  -> 96102-Z-RA-0001-03-1_NEW REVEISON AFTER ADDITIONAL CUSTOMER COMMENTS.DOCX
  -> 32003-J-RA-0012-02A-1_Report.DOCX
  -> 32003-J-RA-0014-01A-1_.DOCX
  -> 500970-A-RA-0002-02A-1_Lesson Learned related to Digital Drilling Upgrade by NOV on Deepsea Stavanger in 2025.DOCX
[Stage 2] Retrieved 50 chunks from 10 document(s)
          Scores: 1.00 (best) → 1.00 (worst)
[Stage 2b] Knowledge graph context: 478 entity entries
[Stage 2b] Community context: 9 community summaries
[Stage 2c] Cypher: 13 row(s)
          Query: MATCH (d:Document)-[:HAS_SECTION]->(:Section)-[:HAS_CHUNK]->(c:Chunk) WHERE d.status = 'current' AND (toLower(d.title) CONTAINS 'drift-off analysis'